# Food Delivery Data Cleaning | Python Project

**Tools:** Python · Pandas · NumPy · Excel  
**Project type:** Data cleaning and preprocessing  
**Output:** A cleaned Excel workbook ready for analysis and dashboard development

---

## 1. Project overview

This notebook prepares a food-delivery dataset for analysis. It checks the dataset structure, removes exact duplicate records, standardizes dates and times, cleans numeric columns, handles missing values, and exports a cleaned Excel file.

### Cleaning principles
- Keep original business statuses unless the source data supports a correction.
- Do not assume that an order in `Preparing` or `Out For Delivery` is `Delivered`.
- Do not guess payment outcomes from an order status.
- Keep missing numeric values as missing unless a defensible imputation rule is available.
- Preserve cancelled orders so cancellation analysis remains possible.

## 2. Import libraries and load the dataset

**Before running:** place the source Excel file in the same folder as this notebook, or update `input_file` with its full path.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

input_file = Path("Food_Dashboard.xlsx")
output_file = Path("Cleaned_Food_Delivery_Data.xlsx")

if not input_file.exists():
    raise FileNotFoundError(
        f"Could not find {input_file}. Place the Excel file beside this notebook "
        "or update input_file with the correct path."
    )

df = pd.read_excel(input_file)

print(f"Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
display(df.head())

## 3. Initial data assessment

Review column names, data types, missing values, and duplicate records before cleaning.

In [ ]:
print("Dataset structure:")
df.info()

print("\nMissing values by column:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))

print("\nExact duplicate rows:", df.duplicated().sum())

## 4. Remove exact duplicate rows

Only fully duplicated rows are removed. Similar-looking records with different values are retained for review.

In [ ]:
rows_before = len(df)
df = df.drop_duplicates().copy()
print(f"Removed {rows_before - len(df):,} exact duplicate rows.")

## 5. Standardize date and time fields

Dates are parsed into pandas datetime values. Time columns are converted to a consistent `HH:MM:SS` text format for export. Unparseable values become missing rather than being guessed.

In [ ]:
if "order_date" in df.columns:
    df["order_date"] = pd.to_datetime(df["order_date"], format="mixed", errors="coerce")

for col in ["order_time", "delivered_at"]:
    if col in df.columns:
        parsed = pd.to_datetime(df[col].astype("string"), errors="coerce")
        df[col] = parsed.dt.strftime("%H:%M:%S")

## 6. Clean numeric columns

The following fields are expected to contain numeric values. The extraction step handles values that contain currency symbols, units, or other surrounding text. Values that cannot be parsed become missing.

In [ ]:
numeric_columns = [
    "item_count", "order_subtotal_inr", "discount_inr", "delivery_fee_inr",
    "tax_inr", "tip_inr", "order_total_inr", "distance_km",
    "prep_time_min", "delivery_time_min", "order_rating"
]

for col in numeric_columns:
    if col in df.columns:
        extracted = df[col].astype("string").str.extract(r"(-?\d+\.?\d*)", expand=False)
        df[col] = pd.to_numeric(extracted, errors="coerce")

print("Numeric conversion complete.")
display(df[[c for c in numeric_columns if c in df.columns]].dtypes.to_frame("data_type"))

## 7. Review invalid numeric values

This project excludes rows where `item_count` or `distance_km` is zero or negative, matching the original cleaning workflow. Missing values are not automatically treated as zero.

In [ ]:
positive_value_columns = [c for c in ["item_count", "distance_km"] if c in df.columns]

for col in positive_value_columns:
    before = len(df)
    df = df[df[col].isna() | (df[col] > 0)].copy()
    print(f"{col}: removed {before - len(df):,} rows with zero or negative values.")

## 8. Standardize text fields and handle missing values

Text columns are trimmed to remove leading/trailing spaces. Missing values are filled with `Unknown` for general text fields, while `promo_code` and `customer_note` receive field-specific labels.

**Note:** `Unknown`, `No_Promo_Code`, and `No_Note` are labels, not recovered original values.

In [ ]:
for col in df.select_dtypes(include=["object", "string"]).columns:
    df[col] = df[col].astype("string").str.strip()

if "promo_code" in df.columns:
    df["promo_code"] = df["promo_code"].fillna("No_Promo_Code")

if "customer_note" in df.columns:
    df["customer_note"] = df["customer_note"].fillna("No_Note")

for col in df.select_dtypes(include=["object", "string"]).columns:
    df[col] = df[col].fillna("Unknown")

## 9. Validate order and payment statuses

Status values are standardized for whitespace and capitalization, but their business meaning is not changed.

For example, `Preparing` and `Out For Delivery` are **not** converted to `Delivered`. Likewise, payment status is not inferred from order status. Review unusual categories with the source system or business owner before correcting them.

In [ ]:
for col in ["order_status", "payment_status"]:
    if col in df.columns:
        df[col] = df[col].astype("string").str.strip().str.title()
        print(f"\n{col} categories:")
        display(df[col].value_counts(dropna=False).to_frame("count"))

## 10. Handle missing order ratings

Missing ratings are filled with the dataset mean, rounded to one decimal place. This is a simple imputation choice; for formal analysis, compare results with and without imputed ratings and document the approach.

In [ ]:
if "order_rating" in df.columns:
    rating_mean = df["order_rating"].mean()
    if pd.notna(rating_mean):
        df["order_rating"] = df["order_rating"].fillna(round(rating_mean, 1))

## 11. Final data-quality check

Compare the final shape, missing values, duplicate count, and data types.

In [ ]:
print(f"Final dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("Remaining exact duplicates:", df.duplicated().sum())

quality_report = pd.DataFrame({
    "data_type": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
})
display(quality_report)

## 12. Export the cleaned dataset

The output workbook is saved in the same folder as this notebook.

In [ ]:
df.to_excel(output_file, index=False)
print(f"Cleaned file saved to: {output_file.resolve()}")

## 13. Project summary

The workflow covers:
- Duplicate removal
- Date and time standardization
- Numeric data conversion
- Invalid value review
- Text standardization and missing-value handling
- Status validation without unsupported assumptions
- Final quality checks and Excel export

**Portfolio note:** Include the source dataset only if you have permission to share it. If it contains personal or sensitive information, use an anonymized sample or omit it from the public repository.